# S11 · De las fotografías a evidencia verificable

**Universidad Central · Big Data · laboratorio de investigación (60 min)**

Laura recibe **actas SECOP**, **fotografías de El Tiempo** y una **voz humana licenciada**. Queremos responder qué se lee, qué se observa y qué se escucha, sin confundir herramientas ni inventar relaciones entre fuentes.

1. ¿Cuál es el TOTAL, el DESCUENTO y el NETO A PAGAR que aparecen en un acta?
2. ¿Qué letras aparecen en los anuncios de una fotografía?
3. ¿Qué fotografía se recupera con la consulta «viviendas en una ladera»?
4. ¿Qué dijo una voz humana y cómo lo comprobamos contra el audio?
5. ¿Cómo preservamos archivo, modalidad y huella SHA-256?

**Lo importante es aprender a escribir código**, no ejecutar un programa terminado. Cada celda introduce pocas líneas; debes editar variables, probar otras imágenes y explicar un resultado o error. Puedes borrar una celda y reescribirla.

**Fuentes separadas:** las fotos periodísticas no prueban nada sobre las actas; la voz MINDS-14 es bancaria y NO pertenece a las noticias. Las actas pueden contener datos personales; evita publicar OCR completo de nombres o números de cuenta.

## 1 · Entorno y archivos (0–6 min)

**Qué es cada herramienta:** Python ejecuta nuestras instrucciones; Pillow lee píxeles; Path representa rutas de archivos; Tesseract (instalado en Linux) reconoce caracteres. La instalación manual se explica en `INSTALACION_UBUNTU.md`. En Codespaces ya está el programa y sus paquetes, pero debes comprobarlos.

**En la terminal:** `python --version` y `tesseract --list-langs`. Debe aparecer `spa`. Luego escribe estas líneas en una celda vacía.

**Tu cambio:** sustituye la foto `006` por `009` y observa la diferencia.

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display, Audio
import hashlib, io, json, re

RAIZ = Path.cwd()
FOTOS = RAIZ / "datos/raw/fotos"
ACTAS = RAIZ / "datos/raw/actas"
SALIDAS = RAIZ / "outputs"
SALIDAS.mkdir(exist_ok=True)

def sha256(ruta):
    return hashlib.sha256(Path(ruta).read_bytes()).hexdigest()

foto = FOTOS / "eltiempo_006.webp"
imagen = Image.open(foto)
print("Formato:", imagen.format, "Píxeles:", imagen.size)
display(imagen)

## 2 · OCR documental con Tesseract (6–18 min)

**OCR:** reconocimiento óptico de caracteres. El programa Tesseract identifica formas de letras y números en la imagen. `pytesseract` llama a ese ejecutable desde Python. No son dos motores diferentes.

**PSM:** modo de segmentación de página. `--psm 6` asume un bloque de texto; `--psm 11` busca fragmentos dispersos.

**Demostración:** abre el acta original y localiza visualmente las columnas de totales. No compartas en capturas nombres, números de identificación o cuentas de terceros.

In [ ]:
import pytesseract

acta = ACTAS / "secop_acta_728490281_pagina1.png"
pagina = Image.open(acta)
# Para proyectar no mostramos nombres ni cuentas: enfocamos solo la zona de totales.
recorte = pagina.crop((0, int(pagina.height * 0.35),
                       pagina.width, int(pagina.height * 0.79)))
display(recorte.resize((700, int(recorte.height * 700 / recorte.width))))

ocr_6 = pytesseract.image_to_string(pagina, lang="spa", config="--psm 6")
print("Líneas con valores:")
for linea in ocr_6.splitlines():
    if any(palabra in linea.upper() for palabra in ("TOTAL", "DESCUENTO", "PAGAR")):
        print(linea.strip())

### ¿Reconoció bien la tabla?

En esta página de ejemplo los importes visibles son TOTAL: 2.200.000; DESCUENTOS: 154.000; NETO A PAGAR: 2.046.000 COP. El cálculo aritmético se puede comprobar. **No se debe asumir que Tesseract leerá todas las cifras correctamente.**

**Tu experimento:** cambia la lista `modos` y compara cómo se recuperan columnas. Si OCR no leyó un importe, muéstralo como error en lugar de cambiar el original. Separa **lo visto por una persona** de **lo predicho por el OCR**.

In [ ]:
modos = [6, 11]  # CAMBIA: agrega 3, compara
for psm in modos:
    texto = pytesseract.image_to_string(pagina, lang="spa", config=f"--psm {psm}")
    candidatos = [x.strip() for x in texto.splitlines()
                  if "TOTAL" in x.upper() or "PAGAR" in x.upper()]
    print("PSM", psm, "candidatos:", candidatos[:4])

total, descuentos, neto = 2200000, 154000, 2046000
print("Comprobación visual + aritmética:", total - descuentos == neto)

## 3 · OCR sobre fotografías: cuándo sirve y cuándo no (18–25 min)

- `eltiempo_006.webp`: taquillas con letreros.
- `eltiempo_023.webp`: nombre de una lotería.
- `eltiempo_001.jpg`: medicamentos y atención sanitaria.
- `eltiempo_009.webp`: viviendas en una ladera.

**Pregunta:** ¿Reconocer texto en una imagen permite afirmar qué está ocurriendo? **No.** La fotografía puede carecer de caracteres y contener información visual relevante.

**Tu cambio:** sustituye `--psm 11` por `--psm 6` y observa qué cambia; identifica un falso positivo.

In [ ]:
nombres = ["eltiempo_006.webp", "eltiempo_023.webp",
           "eltiempo_001.jpg", "eltiempo_009.webp"]
textos_fotos = {}
for nombre in nombres:
    ruta = FOTOS / nombre
    leido = pytesseract.image_to_string(Image.open(ruta),
                                        lang="spa", config="--psm 11").strip()
    textos_fotos[nombre] = leido
    print(nombre, "→", repr(leido[:130]))

## 4 · Búsqueda por significado de fotografías (25–35 min)

**Problema:** «¿Qué foto muestra viviendas en una ladera?». Tesseract no describe escenas. Utilizaremos el modelo `embeddinggemma-2:440m`, que acepta texto e imágenes en Ollama. El modelo `270m` **solo acepta texto**, no imágenes.

**Instalación explicada en directo:** abre `INSTALACION_UBUNTU.md`; desde la terminal verifica `ollama --version` y ejecuta `ollama pull embeddinggemma-2:440m`. Después `curl http://127.0.0.1:11434/api/tags`. Descarga el modelo ANTES de grabar para no ocupar tiempo de clase.

**Definiciones:** Base64 transporta bytes en JSON; `/api/embed` devuelve un vector numérico (768 valores); coseno mide cercanía entre dos vectores. El resultado es un ranking, no una respuesta certificada. El modelo no genera un párrafo de descripción.

In [ ]:
import requests, base64, math

MODELO = "embeddinggemma-2:440m"
API = "http://127.0.0.1:11434/api/embed"

def vector(entrada):
    respuesta = requests.post(API, json={"model": MODELO, "input": entrada},
                              timeout=240)
    respuesta.raise_for_status()
    return respuesta.json()["embeddings"][0]

vectores_fotos = {}
for nombre in nombres:
    contenido = (FOTOS / nombre).read_bytes()
    base64_imagen = base64.b64encode(contenido).decode("ascii")
    vectores_fotos[nombre] = vector({"image": base64_imagen})
    print(nombre, "→", len(vectores_fotos[nombre]), "números")
assert all(len(v) == 768 for v in vectores_fotos.values())

### Tu búsqueda: cambia la pregunta

**Conceptos:** producto punto y norma de un vector. El coseno divide el producto punto entre las normas. Este cálculo es pequeño y comprensible; no hace falta una base de datos vectorial para cuatro imágenes.

Prueba: «viviendas en una ladera», «medicamentos», «taquillas de transporte» y «sorteo de lotería». **Evalúa manualmente** si el primer resultado corresponde a la imagen, sin afirmar precisión universal.

In [ ]:
def coseno(a, b):
    producto = sum(x*y for x, y in zip(a, b))
    norma_a = math.sqrt(sum(x*x for x in a))
    norma_b = math.sqrt(sum(y*y for y in b))
    return producto / (norma_a * norma_b)

pregunta = "viviendas en una ladera"  # TU TURNO: modifica la consulta
consulta = vector("task: search result | query: " + pregunta)
ranking = sorted(((nombre, coseno(consulta, valores))
                  for nombre, valores in vectores_fotos.items()),
                 key=lambda fila: -fila[1])
for nombre, similitud in ranking:
    print(f"{similitud:.3f} · {nombre}")
display(Image.open(FOTOS / ranking[0][0]))

## 5 · Transcribir voz REAL con Faster-Whisper (35–44 min)

**Fuente autorizada:** PolyAI MINDS-14, es-ES, licencia CC BY 4.0. Son conversaciones bancarias, **NO audios de las fotos de El Tiempo**.

Esta celda descarga el **Parquet** de fuente (~39 MB) desde Hugging Face, verifica la huella SHA-256 del corpus y conserva el WAV original de una persona. El audio puede contener el códec G.711: **WAV es un contenedor, no un códec**.

Antes de hablar del ASR, **escucha** el WAV. La transcripción `referencia` pertenece al corpus, no es la predicción de Whisper. Si falla la descarga o el hash, detente: no fabriques datos de sustitución.

In [ ]:
from urllib.request import Request, urlopen
import pandas as pd

URL = ("https://huggingface.co/datasets/PolyAI/minds14/resolve/main/"
       "es-ES/train-00000-of-00001.parquet")
SHA_CORPUS = "989bf2f90676a5eb5852fe785a03b1d9b97763e0febbd0fd0a3f489afd83badf"
corpus = RAIZ / "datos/minds14_es-ES.parquet"

if not corpus.exists():
    peticion = Request(URL, headers={"User-Agent": "BigData-UCentral-educativo/1.0"})
    with urlopen(peticion, timeout=120) as red, corpus.open("wb") as salida:
        while bloque := red.read(1024 * 1024):
            salida.write(bloque)
            if salida.tell() > 100_000_000:
                raise ValueError("Archivo supera los 100 MB")
if sha256(corpus) != SHA_CORPUS:
    raise ValueError("El Parquet de audio no coincide con la fuente auditada")

audios = pd.read_parquet(corpus, columns=["audio", "path", "transcription"])
fila = audios.sort_values("path").iloc[0]
wav = RAIZ / "datos/raw/voz_minds14.wav"
wav.write_bytes(fila["audio"]["bytes"])
referencia = str(fila["transcription"])
print("Fuente:", fila["path"], "| SHA:", sha256(wav)[:16])
display(Audio(filename=str(wav)))

### ¿Qué aprende el modelo Whisper?

**Faster-Whisper** ejecuta `Whisper tiny` en CPU con cuantización `int8`: representación de 8 bits para reducir uso de memoria. El modelo recibe la grabación, no copia la transcripción humana. Devuelve segmentos con **inicio, fin y texto**.

**Tu verificación:** identifica una frase problemática, reproduce el intervalo y compara lo que dijo la persona con lo que predijo el modelo. No ocultes errores.

In [ ]:
from faster_whisper import WhisperModel

modelo_voz = WhisperModel("tiny", device="cpu", compute_type="int8",
                          cpu_threads=2)
segmentos, info = modelo_voz.transcribe(str(wav), language="es", beam_size=2)
segmentos = list(segmentos)
hipotesis = " ".join(s.text.strip() for s in segmentos)
print("Predicción Whisper:", hipotesis)
print("Referencia humana:", referencia)
for s in segmentos:
    print(round(s.start, 1), "→", round(s.end, 1), s.text)

### Medir WER con una función pequeña

**WER:** tasa de error de palabras = número mínimo de inserciones, borrados y sustituciones dividido entre las palabras de referencia. Aquí la referencia humana existe porque viene del corpus. Sin ella **NO** afirmes una tasa de error. El WER solo describe este audio y este método de comparación.

In [ ]:
def palabras(texto):
    return re.findall(r"\w+", texto.lower())

def errores(a, b):
    anterior = list(range(len(b) + 1))
    for i, palabra in enumerate(a, 1):
        actual = [i]
        for j, otra in enumerate(b, 1):
            actual.append(min(actual[-1] + 1, anterior[j] + 1,
                              anterior[j-1] + (palabra != otra)))
        anterior = actual
    return anterior[-1]

ref = palabras(referencia)
pred = palabras(hipotesis)
print("WER de este audio:", round(errores(ref, pred) / len(ref), 3))

## 6 · ETL con Python escrito por ti (44–54 min)

**E = extraer:** abrir original, recuperar texto o voz.  
**T = transformar:** crear diccionarios comparables, conservar tipo y procedencia.  
**L = cargar:** guardar **JSONL** (un objeto por línea) y **Parquet** (columnas tipadas).

No vamos a ejecutar un motor de 400 líneas. Vas a escribir `for`, `dict`, `json.dumps` y `to_parquet`.

**Privacidad:** **no publiques el OCR completo de las actas**: pueden contener identificaciones y cuentas. Guarda solo la referencia al archivo y los metadatos mínimos. Una huella SHA-256 prueba igualdad de bytes, no que la fuente diga la verdad.

In [ ]:
registros = []
for nombre in nombres:
    archivo = FOTOS / nombre
    registros.append({"archivo": str(archivo.relative_to(RAIZ)),
                      "modalidad": "foto_periodistica",
                      "sha256": sha256(archivo),
                      "texto_recortado": textos_fotos[nombre][:100],
                      "credito": "ver ficha editorial"})

registros.append({"archivo": str(acta.relative_to(RAIZ)),
                  "modalidad": "documento_secop", "sha256": sha256(acta),
                  "texto_recortado": "[OCR protegido: datos personales]",
                  "credito": "documento público SECOP, revisar contexto"})
registros.append({"archivo": str(wav.relative_to(RAIZ)),
                  "modalidad": "audio_minds14", "sha256": sha256(wav),
                  "texto_recortado": hipotesis[:100],
                  "credito": "PolyAI MINDS-14 es-ES, CC BY 4.0"})

ruta_jsonl = SALIDAS / "catalogo.jsonl"
with ruta_jsonl.open("w", encoding="utf-8") as salida:
    for dato in registros:
        salida.write(json.dumps(dato, ensure_ascii=False) + "\n")
pd.DataFrame(registros).to_parquet(SALIDAS / "catalogo.parquet", index=False)
display(pd.DataFrame(registros)[["archivo", "modalidad", "sha256"]])

### Verificar datos sin confiar en una salida bonita

**Comprueba:** reabrir JSONL y Parquet; contar filas; comparar los hashes actuales con los que guardaste. Una fuente alterada debe detectarse. Si quieres experimentar, modifica **una copia**, nunca el archivo RAW.

In [ ]:
filas = [json.loads(linea) for linea in ruta_jsonl.read_text().splitlines()]
tabla = pd.read_parquet(SALIDAS / "catalogo.parquet")
assert len(filas) == len(tabla)
for fila in filas:
    assert sha256(RAIZ / fila["archivo"]) == fila["sha256"]
print("OK:", len(filas), "registros, integridad de originales comprobada")

## 7 · Reto final (54–60 min)

Sin ejecutar ningún programa de terceros, **elige un archivo distinto** y modifica al menos tres líneas de Python:

- **Documento:** cambia a la segunda acta y compara PSM 6/11; lee visualmente un valor.
- **Imagen:** cambia pregunta y compara los cuatro resultados de búsqueda; verifica el original.
- **Audio:** toma otro registro autorizado del corpus y vuelve a transcribir, comparando la referencia humana.

**Producto breve:** `pregunta | archivo | herramienta | resultado | cómo lo verifiqué | posible error`.

**Criterio de aprendizaje:** debes poder responder **qué hace cada línea esencial**, por qué OCR no describe escenas, por qué el vector no certifica hechos, cómo se comprueba WER y cómo adaptar el flujo a otro conjunto propio. La mera ejecución de una celda no acredita aprendizaje.

**Referencias:** Tesseract OCR; Faster-Whisper (SYSTRAN); EmbeddingGemma 2 (Google/Ollama); PolyAI MINDS-14 es-ES, CC BY 4.0. Fotos: El Tiempo, créditos pendientes de curaduría por imagen. SECOP: documentos públicos, evita difundir datos personales.